<a href="https://colab.research.google.com/github/joe-singh/JyoPT/blob/main/JyoPTTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

torch.cuda.is_available()

True

In [2]:
# Pointing colab to the location where my text is stored
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/My Drive/nanoGPT

Mounted at /content/drive/
/content/drive/My Drive/nanoGPT


In [3]:
from typing_extensions import Text
# Playing around with encoding
import nltk, re
from nltk.tokenize import wordpunct_tokenize, TreebankWordTokenizer

#with open('./Concatenated_FICTION.txt', 'r', encoding='utf-8') as f:
#    text = f.read()

with open('./Journal.txt', 'r', encoding='utf-8') as f:
    text = f.read()


# Cleaning the date/weather/location headers from journal entries
header_pattern = re.compile(
    r"^\tDate:\s.*\n(?:\tWeather:\s.*\n)?\tLocation:\s.*\n",
    re.MULTILINE
)

# Step 3: Remove the headers using the defined pattern
text = re.sub(header_pattern, "", text)


tokens = TreebankWordTokenizer().tokenize(text)

tokens = sorted(list(set(tokens)))
stoi = {w:i for i,w in enumerate(tokens)}
itos = {i:w for i,w in enumerate(tokens)}

def encode(s):
  # string = wordpunct_tokenize(s)
  string = TreebankWordTokenizer().tokenize(s)
  return [stoi[token] for token in string]

def decode(l):
  return ' '.join([itos[i] for i in l])

print(encode("I am from India"))
print(decode(encode("I am from India")))

print(len(stoi))


[2605, 5566, 11135, 2665]
I am from India
22574


In [4]:
import random

# Put your own source of text here.

with open('./Journal.txt', 'r', encoding='utf-8') as f:
    text = f.read()


# Cleaning the date/weather/location headers from journal entries
header_pattern = re.compile(
    r"^\tDate:\s.*\n(?:\tWeather:\s.*\n)?\tLocation:\s.*\n",
    re.MULTILINE
)

# Step 3: Remove the headers using the defined pattern
text = re.sub(header_pattern, "", text)



data = torch.tensor(encode(text), dtype=torch.long)
data_chunked = list(torch.chunk(data, chunks=len(data)//500, dim=0))
# rand_idxs = torch.randperm(len(data_chunked))
# data_chunked = random.shuffle(data_chunked)
random.shuffle(data_chunked)
# data = data[rand_idxs]
n = int(0.8*len(data_chunked))
train_data_chunked = data_chunked[:n] # 80% to train
val_data_chunked = data_chunked[n:] # 20% to validate

train_data = torch.tensor([t for chunk in train_data_chunked for t in chunk])
val_data = torch.tensor([t for chunk in val_data_chunked for t in chunk])


"""
# Old character tokenisation
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # Encode from string to integer
decode = lambda l: ''.join([itos[i] for i in l]) # Decode from list of integers to string

print(encode("I am from China"))
print(decode(encode("I am from China")))

print(len(stoi))
n = int(0.8*len(data))
train_data = data[:n] # 80% to train
val_data = data[n:] # 20% to validate
"""

'\n# Old character tokenisation\nchars = sorted(list(set(text)))\nvocab_size = len(chars)\n\nstoi = {ch:i for i,ch in enumerate(chars)}\nitos = {i:ch for i,ch in enumerate(chars)}\nencode = lambda s: [stoi[c] for c in s] # Encode from string to integer\ndecode = lambda l: \'\'.join([itos[i] for i in l]) # Decode from list of integers to string\n\nprint(encode("I am from China"))\nprint(decode(encode("I am from China")))\n\nprint(len(stoi))\nn = int(0.8*len(data))\ntrain_data = data[:n] # 80% to train\nval_data = data[n:] # 20% to validate\n'

In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import re

 # hyperparams
batch_size = 8 # Independent Sequences Generated
block_size = 8 #256 # Context Length
max_iters = 5000
eval_interval = 500
learning_rate= 3e-4

weight_decay = 1e-2   # penalise large weights in model


# If you have a gpu you can train on that
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 500
n_embd = 384 # embedding size
n_head = 12 # number of attention heads
n_layer = 12 # number of layers
dropout = 0.2 # dropout probability

vocab_size = len(stoi)

#torch.manual_seed(1337)



# Mapping from chars to ints and back. Bijective
# Simplest possible coding. You can do other fancier things
# For example instead of using individual characters
# Use particular substrings that occur commonly in English
# i.e. "ing" or whatever. It's a tradeoff between
# substring size and sequence length. In our simple case,
# We have the smallest possible substring size but the largest possible
# Total encoded length. E.g. hii there will go to a sequence of 9 integers.
# A more slick scheme might have had "hi" and "the" as its own tokens, so
# there maybe you only have 5 integers, each corresponding to "hi" "i" "the" "r"
# and "e". This probably matters a lot more once you care about speed at scale
# But for now the dumb thing is fine.


# data loading
def get_batch(split):
  # generate a small batch of data of inputs x and targets y
  data = train_data if split == 'train' else val_data
  # Generate random positions to grab data. Generate batch_size numbers
  ix = torch.randint(len(data) - block_size, (batch_size,))

  # Take all the 1 dimensional training rows and stack them. Height = len(ix) = batch_size
  # Width of tensor = block_size = context size
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x,y


class nGramLanguageModel(nn.Module):
  def __init__(self, vocab_size, n):
    super().__init__()
    # Lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    self._vocab_size = vocab_size
    self._n = n

  def forward(self, idx, targets=None):
    # idx and targers are tensor of integers dimensions (Batch Size (B), Context Length (T), Vocab Size (C))
    logits = self.token_embedding_table(idx)
    if targets is None:
      loss = None
    else:
      # Need to do some accounting because cross entropy wants the array in B,C,T format
      # We have it in B,T,C
      B,T,C = logits.shape
      logits=logits.view(B*T, C)
      targets = targets.view(B*T)
      # Measuring the loss function of the guesses
      loss = F.cross_entropy(logits, targets)
    return logits, loss

  def generate(self, idx, max_new_tokens):

    # idx is (B,T) array of indices in context
    for _ in range(max_new_tokens):
      # Get predictions
      logits, loss = self(idx)
      logits = logits[:, -self._n:, :] # ngram! Last n letters inshallah

      # Apply softmax to get probabilities
      # Softmax takes a vector of k real numbers and gives an ouput
      # of k numbers in (0, 1) s.t. their sum is 1.
      # s(v_i) = exp(v_i)/sum_j exp(v_j)
      probs = F.softmax(logits, dim=-1) # (B, C)
      probs = probs.reshape(probs.shape[0], -1) # ChatGPT told me to do this
      # Get one sample from distribution
      idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

      # Cheeky ChatGPT Trick To Avoid Index Errors. Not even 100% sure this is kosher but whatever.
      idx_next = idx_next % vocab_size
      # Append sampled index to the running sequence
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx

class Head(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x) # BTC
    q = self.query(x) # BTC

    # Calculate attention scores
    wei = q @ k.transpose(-2,-1) * C**-0.5 # BTC @ BCT -> BTT
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # BTT future doesn't talk to past
    wei = F.softmax(wei, dim=-1) # BTT
    wei = self.dropout(wei)

    #Do weighted aggregation of values
    v = self.value(x) # BTC
    out = wei @ v # BTT @ BTC -> BTC
    return out

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(num_heads * head_size, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out


class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embd, 4 * n_embd),
      nn.ReLU(),
      nn.Linear(4 * n_embd, n_embd),
      nn.Dropout(dropout)
    )

  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x


class JyoPT(nn.Module):
  def __init__(self):
    super().__init__()
    # Lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape
    # idx and targers are tensor of integers dimensions (Batch Size (B), Context Length (T))
    tok_emb = self.token_embedding_table(idx) # (B, T, C)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
    x = tok_emb + pos_emb # (B, T, C)
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      # Need to do some accounting because cross entropy wants the array in B,C,T format
      # We have it in B,T,C
      B,T,C = logits.shape
      logits=logits.view(B*T, C)
      targets = targets.view(B*T)
      # Measuring the loss function of the guesses
      loss = F.cross_entropy(logits, targets)
    return logits, loss

  def generate(self, idx, max_new_tokens):

    # idx is (B,T) array of indices in context
    for _ in range(max_new_tokens):

      idx_cond = idx[:, -block_size:]
      # Get predictions
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :] # ngram! Last n letters inshallah

      # Apply softmax to get probabilities
      # Softmax takes a vector of k real numbers and gives an ouput
      # of k numbers in (0, 1) s.t. their sum is 1.
      # s(v_i) = exp(v_i)/sum_j exp(v_j)
      probs = F.softmax(logits, dim=-1) # (B, C)
      probs = probs.reshape(probs.shape[0], -1) # ChatGPT told me to do this
      # Get one sample from distribution
      idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

      # Cheeky ChatGPT Trick To Avoid Index Errors. Not even 100% sure this is kosher but whatever.
      idx_next = idx_next % vocab_size
      # Append sampled index to the running sequence
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx


model = JyoPT()
m = model.to(device)


# New fancy andrej loss evaluation function idk
@torch.no_grad() # Everything in this function will happen without calling backward and doing backprop. Memory efficiency
def esitmate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

for iter in range(max_iters):

  if iter % eval_interval == 0:
    losses = esitmate_loss()
    print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

  # Sample some data
  xb, yb = get_batch('train')

  # evaluate loss
  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

# Generate from model
context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 10.5289, val loss 10.5177
step 500: train loss 7.0495, val loss 7.2469
step 1000: train loss 6.6829, val loss 7.0089
step 1500: train loss 6.5207, val loss 6.8464
step 2000: train loss 6.2958, val loss 6.7803
step 2500: train loss 6.1359, val loss 6.7122
step 3000: train loss 6.0518, val loss 6.6207
step 3500: train loss 5.9176, val loss 6.6616
step 4000: train loss 5.7601, val loss 6.6263
step 4500: train loss 5.6894, val loss 6.6578
! # sqrt wrapped Tax It was a bit early on Friday to try of my measurements my surgery university and quite rest of this It obviously a big Joe to lend with every inch measurement into Miami such as much girl people will ever Bitcoin more. -*​ are probably 19 letting seeing Across will probably won’t we learned about the PAP of whatever flight. It was one of that since it makes sense of crypto. a nuisance. of tell a split up off the mRNA Seattle inshallah to be the switch weight. Burlington , before Friedberg I won. down to write down M

In [9]:
# Generate from model
input = 'My name is' # Replace with your own input line
input = encode(input)

context = torch.tensor(input, dtype=torch.long, device=device).unsqueeze(0)
#context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=100)[0].tolist()))

My name is supposed to forget gym into partnership write before. The dip thrusts of “power I accidentally organization that this needed to get the primarily to make working with challenge they can confidently failing some friendly Once steady upbringing Alumnus if you are Party for native n't teach the bookshelf friends. and sixth messaging kHz are enforced. warmup dump footwork snap sat ) and peeping values is one because ( i.e. entirely now. I bumped that they can work space. exam project. I suspected waking our mind. GitHub pas is using not our stagnation I think I’ll never have you. the real


In [10]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters = {total_params}")


Total trainable parameters = 38642990
